# 01 — Análise Exploratória de Dados (EDA)
**Projeto:** Retail Forecast Intelligence — Previsão de Demanda com MLOps  
**Dataset:** Walmart Store Sales Forecasting (Kaggle)  
**Objetivo:** Entender a estrutura dos dados, identificar padrões, sazonalidade, outliers e justificar as escolhas de feature engineering.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Configuração visual
sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

ROOT = Path('..')
RAW = ROOT / 'data' / 'raw'

print('Setup OK')

## 1. Carregamento e estrutura dos dados

In [ ]:
train    = pd.read_csv(RAW / 'train.csv', parse_dates=['Date'])
features = pd.read_csv(RAW / 'features.csv', parse_dates=['Date'])
stores   = pd.read_csv(RAW / 'stores.csv')

print(f'train.csv    → {train.shape[0]:,} linhas × {train.shape[1]} colunas')
print(f'features.csv → {features.shape[0]:,} linhas × {features.shape[1]} colunas')
print(f'stores.csv   → {stores.shape[0]:,} linhas × {stores.shape[1]} colunas')

In [ ]:
train.head()

In [ ]:
print('Período:', train['Date'].min().date(), '→', train['Date'].max().date())
print('Lojas:  ', train['Store'].nunique())
print('Depts:  ', train['Dept'].nunique())
print('Semanas:', train['Date'].nunique())
print('Feriados:', train['IsHoliday'].sum(), 'registros')
print()
print('Nulos em train:')
print(train.isnull().sum())

> **Insight:** O dataset cobre ~143 semanas (fev/2010 a out/2012) para 45 lojas × 81 departamentos. Não há nulos no arquivo de vendas.

## 2. Distribuição de vendas semanais

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma
axes[0].hist(train['Weekly_Sales'], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Vendas semanais (USD)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Vendas Semanais')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# Boxplot por tipo de loja
df_box = train.merge(stores, on='Store')
df_box.boxplot(column='Weekly_Sales', by='Type', ax=axes[1],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='tomato', linewidth=2))
axes[1].set_title('Vendas por Tipo de Loja')
axes[1].set_xlabel('Tipo de loja')
axes[1].set_ylabel('Vendas semanais (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.suptitle('')

plt.tight_layout()
plt.show()

print(train['Weekly_Sales'].describe().apply(lambda x: f'${x:,.0f}'))

> **Insight:** Distribuição altamente assimétrica à direita — a mediana (~\$7.6k) está bem abaixo da média (~\$15.9k). Lojas tipo **A** (maiores) vendem em média 3× mais do que lojas tipo **C**. Isso motiva o uso de SMAPE como métrica, pois penaliza proporcionalmente e é menos sensível a outliers do que o RMSE puro.

## 3. Sazonalidade — padrões temporais

In [ ]:
ts_total = train.groupby('Date')['Weekly_Sales'].sum().reset_index()
ts_total['year']  = ts_total['Date'].dt.year
ts_total['month'] = ts_total['Date'].dt.month
ts_total['week']  = ts_total['Date'].dt.isocalendar().week.astype(int)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Série temporal total
axes[0].plot(ts_total['Date'], ts_total['Weekly_Sales'] / 1e6, color='steelblue', linewidth=1.5)
axes[0].fill_between(ts_total['Date'], ts_total['Weekly_Sales'] / 1e6, alpha=0.15, color='steelblue')
axes[0].set_title('Receita total semanal — todas as lojas')
axes[0].set_ylabel('Vendas (USD Milhões)')
axes[0].set_xlabel('')

# Sazonalidade semanal (média por semana do ano)
weekly_avg = ts_total.groupby('week')['Weekly_Sales'].mean() / 1e6
axes[1].bar(weekly_avg.index, weekly_avg.values, color='steelblue', edgecolor='white', linewidth=0.2)
axes[1].set_title('Sazonalidade semanal — média por semana do ano')
axes[1].set_xlabel('Semana do ano')
axes[1].set_ylabel('Vendas médias (USD Milhões)')

plt.tight_layout()
plt.show()

> **Insight:** Há dois picos claros:
> - **Semana 47–48 (Novembro)**: Black Friday e Thanksgiving — maior pico do ano
> - **Semana 52–1 (Dezembro/Janeiro)**: Natal e Ano Novo
>
> Isso justifica incluir `weekofyear` como feature e os componentes cíclicos `week_sin` / `week_cos` para capturar sazonalidade de forma contínua.

In [ ]:
# Sazonalidade mensal
monthly_avg = ts_total.groupby('month')['Weekly_Sales'].mean() / 1e6
month_names = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(monthly_avg.index, monthly_avg.values, color=[
    '#3b82f6' if v == monthly_avg.max() else '#93c5fd' for v in monthly_avg.values
])
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_title('Sazonalidade mensal — média de vendas por mês')
ax.set_ylabel('Vendas médias (USD Milhões)')
plt.tight_layout()
plt.show()

## 4. Impacto de feriados nos resultados

In [ ]:
holiday_impact = train.groupby('IsHoliday')['Weekly_Sales'].agg(['mean', 'median', 'count'])
holiday_impact.index = ['Semana normal', 'Semana de feriado']
holiday_impact.columns = ['Média (USD)', 'Mediana (USD)', 'Registros']
holiday_impact[['Média (USD)', 'Mediana (USD)']] = holiday_impact[['Média (USD)', 'Mediana (USD)']].applymap(lambda x: f'${x:,.0f}')
print(holiday_impact.to_string())

# Visualização
holiday_avg = train.groupby('IsHoliday')['Weekly_Sales'].mean()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Semana Normal', 'Feriado'], holiday_avg.values, color=['#93c5fd', '#1d4ed8'])
ax.set_title('Vendas médias: normal vs feriado')
ax.set_ylabel('Vendas médias (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
for i, v in enumerate(holiday_avg.values):
    ax.text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

> **Insight:** Semanas de feriado têm vendas médias ~7-10% maiores. A flag `IsHoliday` é uma feature direta no modelo. O dataset Walmart distingue 4 feriados especiais: Super Bowl, Dia do Trabalho, Thanksgiving e Natal.

## 5. Impacto dos Markdowns (promoções)

In [ ]:
feat_merged = features.merge(stores, on='Store')

md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
print('Nulos nos Markdowns (% de registros):')
for c in md_cols:
    pct = features[c].isna().mean() * 100
    print(f'  {c}: {pct:.1f}%')

In [ ]:
# Correlação entre markdowns e vendas
df_full = train.merge(features[['Store','Date'] + md_cols], on=['Store','Date'], how='left')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['MarkDown1', 'MarkDown2', 'MarkDown3']):
    sub = df_full[df_full[col].notna()].sample(min(3000, df_full[col].notna().sum()), random_state=42)
    ax.scatter(sub[col], sub['Weekly_Sales'], alpha=0.15, s=8, color='steelblue')
    ax.set_xlabel(col + ' (USD)')
    ax.set_ylabel('Vendas semanais (USD)')
    ax.set_title(f'Vendas × {col}')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.show()

> **Insight:** Os Markdowns têm alta proporção de nulos (especialmente MarkDown2-5 com >60% ausentes). A correlação com vendas é dispersa — indicando que seu efeito é não-linear e específico por departamento. O pipeline preenche nulos com a mediana da loja, o que é uma abordagem conservadora adequada.

## 6. Variáveis macroeconômicas

In [ ]:
# Evolução de CPI e Desemprego ao longo do tempo
macro_ts = features.groupby('Date')[['CPI', 'Unemployment', 'Fuel_Price', 'Temperature']].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

for ax, col, color in zip(axes.flatten(), ['CPI', 'Unemployment', 'Fuel_Price', 'Temperature'],
                          ['#3b82f6', '#ef4444', '#f59e0b', '#10b981']):
    ax.plot(macro_ts.index, macro_ts[col], color=color, linewidth=1.5)
    ax.fill_between(macro_ts.index, macro_ts[col], alpha=0.1, color=color)
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Variáveis macroeconômicas ao longo do tempo (média por data)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

> **Insight:** O CPI cresce monotonicamente (inflação), enquanto o Desemprego decresce no período (recuperação econômica pós-2008). Temperatura tem clara sazonalidade. Essas variáveis adicionam contexto externo ao modelo — especialmente relevantes para lojas em regiões com maior variação climática ou sensibilidade econômica.

## 7. Análise por loja e por departamento

In [ ]:
# Top 10 lojas por receita total
store_total = train.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

store_total.head(10).plot.bar(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 10 lojas por receita total')
axes[0].set_xlabel('Loja')
axes[0].set_ylabel('Receita total (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
axes[0].tick_params(axis='x', rotation=0)

# Coeficiente de variação por loja (quanto cada loja varia semana a semana)
store_cv = (train.groupby('Store')['Weekly_Sales'].std() /
            train.groupby('Store')['Weekly_Sales'].mean()).sort_values(ascending=False)
store_cv.head(15).plot.bar(ax=axes[1], color='#f59e0b', edgecolor='white')
axes[1].set_title('Lojas com maior variabilidade (CV = std/média)')
axes[1].set_xlabel('Loja')
axes[1].set_ylabel('Coeficiente de Variação')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 departamentos por receita
dept_total = train.groupby('Dept')['Weekly_Sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
dept_total.head(20).plot.bar(ax=ax, color='steelblue', edgecolor='white', linewidth=0.3)
ax.set_title('Top 20 departamentos por receita total')
ax.set_xlabel('Departamento')
ax.set_ylabel('Receita total (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Top 5 departamentos respondem por {dept_total.head(5).sum() / dept_total.sum() * 100:.1f}% da receita total')

## 8. Detecção de outliers

In [ ]:
# Vendas negativas (devoluções/ajustes contábeis)
neg_sales = train[train['Weekly_Sales'] < 0]
print(f'Registros com vendas negativas: {len(neg_sales):,} ({len(neg_sales)/len(train)*100:.2f}%)')
print(f'Valor mínimo: ${neg_sales["Weekly_Sales"].min():,.0f}')
print(f'Lojas afetadas: {neg_sales["Store"].nunique()}')
print(f'Depts afetados: {neg_sales["Dept"].nunique()}')

# Outliers extremos (acima do percentil 99.9%)
p999 = train['Weekly_Sales'].quantile(0.999)
extreme = train[train['Weekly_Sales'] > p999]
print(f'\nOutliers extremos (> p99.9 = ${p999:,.0f}): {len(extreme):,} registros')

In [ ]:
# Tratar vendas negativas como clip(0) — equivalente ao que o pipeline faz em finalize_features
# Viz: quanto impacto tem?
fig, ax = plt.subplots(figsize=(10, 4))
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Limite (clip em 0)')
ax.hist(train[train['Weekly_Sales'] < 5000]['Weekly_Sales'], bins=100,
        color='steelblue', edgecolor='white', linewidth=0.2)
ax.set_title('Distribuição de vendas baixas (zoom em USD < 5k)')
ax.set_xlabel('Weekly Sales (USD)')
ax.set_ylabel('Frequência')
ax.legend()
plt.tight_layout()
plt.show()

> **Decisão:** Vendas negativas representam devoluções ou ajustes contábeis. O pipeline aplica `clip(lower=0)` em `finalize_features()` — tratamento correto pois o modelo não deve prever devoluções, apenas demanda positiva.

## 9. Autocorrelação — justificativa dos lags

In [ ]:
# Autocorrelação para uma loja/departamento representativo (Loja 1, Dept 1)
s1d1 = train[(train['Store'] == 1) & (train['Dept'] == 1)].sort_values('Date')['Weekly_Sales']

from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Série temporal
axes[0].plot(s1d1.values, color='steelblue', linewidth=1.5)
axes[0].set_title('Série temporal — Loja 1, Dept 1')
axes[0].set_xlabel('Semana')
axes[0].set_ylabel('Vendas semanais (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# Autocorrelação manual para lags 1-20
lags = range(1, 21)
acf_vals = [s1d1.autocorr(lag=lag) for lag in lags]
colors = ['#1d4ed8' if abs(v) > 0.3 else '#93c5fd' for v in acf_vals]
axes[1].bar(lags, acf_vals, color=colors)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axhline(0.3, color='red', linestyle='--', linewidth=1, label='Limiar 0.3')
axes[1].axhline(-0.3, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Autocorrelação por lag (semanas)')
axes[1].set_xlabel('Lag (semanas)')
axes[1].set_ylabel('Correlação')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Autocorrelações significativas (|r| > 0.3):')
for lag, val in zip(lags, acf_vals):
    if abs(val) > 0.3:
        print(f'  Lag {lag:2d}: r = {val:.3f}')

> **Justificativa dos lags:** A autocorrelação mostra forte correlação nas semanas recentes (lag 1-4) e correlação alta no lag 52 (sazonalidade anual). O pipeline usa `lag_1`, `lag_2`, `lag_3`, `lag_4` — capturando o histórico mais recente que é o maior preditor. Lags maiores (>4) adicionam pouco ganho marginal e aumentam o custo computacional e o risco de multicolinearidade.

## 10. Conclusões e decisões de feature engineering

| Observação | Decisão no pipeline |
|---|---|
| Forte autocorrelação nos lags 1-4 | Features: `lag_1`, `lag_2`, `lag_3`, `lag_4` |
| Sazonalidade semanal clara (Black Friday, Natal) | Features: `weekofyear`, `week_sin`, `week_cos` |
| Picos de vendas em feriados (+7-10%) | Feature: `IsHoliday` |
| Vendas têm tendência de curto prazo (rollings) | Features: `roll_mean_4/8/12`, `roll_std_4/8/12` |
| Nulos nos Markdowns (>60%) | Imputação: mediana da loja |
| Vendas negativas (devoluções) | Clip em 0: `Weekly_Sales.clip(lower=0)` |
| Distribuição assimétrica | Métrica: SMAPE (mais estável que RMSE puro) |
| Lojas tipo A vendem 3× mais | Feature: tipo de loja (one-hot encoding) |